# Detectron2 Mask R-CNN Hyperparameter Tuning

This notebook demonstrates sequential hyperparameter tuning for Detectron2 using Optuna. It maps the conceptual search space used in our YOLO trials into the Detectron2 ecosystem. Due to framework differences, augmentations and loss weights are adapted to Native Detectron2 configurations where possible.

In [ ]:
import os
import gc
import torch
import optuna
import random
import string
from detectron2.config import get_cfg
from detectron2 import model_zoo
from detectron2.engine import DefaultTrainer
from detectron2.evaluation import COCOEvaluator, inference_on_dataset
from detectron2.data import build_detection_test_loader
from detectron2.data.datasets import register_coco_instances

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

DATASET_ROOT = "../../datasets/batch4_coco"

# Register datasets if not already registered
try:
    register_coco_instances("rock_train_optuna", {}, os.path.join(DATASET_ROOT, "train/annotations/instances_default.json"), os.path.join(DATASET_ROOT, "train/images"))
    register_coco_instances("rock_val_optuna", {}, os.path.join(DATASET_ROOT, "val/annotations/instances_default.json"), os.path.join(DATASET_ROOT, "val/images"))
except Exception as e:
    pass # Already registered

def generate_runner_name(prefix="D2_Tuning"):
    suffix = "".join(random.choices(string.ascii_letters + string.digits, k=11))
    return f"{prefix}_{suffix}"

TUNE_NAME = generate_runner_name()

In [ ]:
def run_optuna_tuning():
    print("\n" + "=" * 80)
    print("STARTING HYPERPARAMETER TUNING (OPTUNA) FOR DETECTRON2")
    print("=" * 80 + "\n")

    TUNE_ITERATIONS = 10
    
    def objective(trial):
        print(f"\nStarting Trial {trial.number}")
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            
        # Config Initialization
        cfg = get_cfg()
        cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))
        cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml")
        cfg.DATASETS.TRAIN = ("rock_train_optuna",)
        cfg.DATASETS.TEST = ("rock_val_optuna",)
        cfg.DATALOADER.NUM_WORKERS = 2
        
        # Search Space Mapping (Analogous to YOLO)
        cfg.SOLVER.BASE_LR = trial.suggest_float("lr", 1e-5, 1e-3, log=True)
        cfg.SOLVER.WEIGHT_DECAY = trial.suggest_float("weight_decay", 0.0, 0.001)
        cfg.MODEL.ROI_HEADS.BATCH_SIZE_PER_IMAGE = trial.suggest_categorical("roi_batch_size", [128, 256, 512])
        cfg.MODEL.RPN.BBOX_REG_LOSS_WEIGHT = trial.suggest_float("box_weight", 0.5, 2.0)
        
        # Note: Mask/Cls specific weighting and HSV augmentations require custom Detectron2 mappers/heads.
        # For tuning within the default standard architecture, we tune RPN and Solver equivalents.
        
        # Memory Safe Limits for Tuning
        cfg.SOLVER.IMS_PER_BATCH = 2
        cfg.SOLVER.MAX_ITER = 300
        cfg.MODEL.ROI_HEADS.NUM_CLASSES = 6
        cfg.OUTPUT_DIR = f"./models/{TUNE_NAME}/trial_{trial.number}"
        os.makedirs(cfg.OUTPUT_DIR, exist_ok=True)
        
        try:
            trainer = DefaultTrainer(cfg)
            trainer.resume_or_load(resume=False)
            trainer.train()
            
            evaluator = COCOEvaluator("rock_val_optuna", output_dir=cfg.OUTPUT_DIR)
            val_loader = build_detection_test_loader(cfg, "rock_val_optuna")
            metrics = inference_on_dataset(trainer.model, val_loader, evaluator)
            
            if "segm" in metrics and "AP" in metrics["segm"]:
                metric = metrics["segm"]["AP"]
            else:
                metric = 0.0
                
            return metric
            
        except Exception as e:
            print(f"Error during Trial {trial.number}: {str(e)}")
            raise optuna.exceptions.TrialPruned()
            
        finally:
            del trainer
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

    study = optuna.create_study(direction="maximize", study_name=TUNE_NAME)
    study.optimize(objective, n_trials=TUNE_ITERATIONS, gc_after_trial=True)
    
    print("\n" + "=" * 80)
    print("OPTUNA TUNING COMPLETED!")
    print("=" * 80)
    print("Best Hyperparameters:")
    for key, value in study.best_params.items():
        print(f"  - {key}: {value}")

    return study.best_params

best_optuna_results = run_optuna_tuning()